# Financial PhraseBank — Dataset Audit

## Goal

Load and inspect the Financial PhraseBank dataset before modelling.

Tasks:
1. Compare annotator-agreement configurations
2. Select an agreement threshold
3. Inspect class distribution
4. Identify missing values
5. Identify duplicates and conflicting labels
6. Perform minimal cleaning
7. Produce the dataset used for splitting

In [2]:
!pip install -q datasets pandas matplotlib scikit-learn

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import zipfile
import pandas as pd
from huggingface_hub import hf_hub_download

# Download/cache the Financial PhraseBank ZIP
zip_path = hf_hub_download(
    repo_id="takala/financial_phrasebank",
    repo_type="dataset",
    filename="data/FinancialPhraseBank-v1.0.zip"
)

# Sentiment label mapping
label_names = ["negative", "neutral", "positive"]
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

In [5]:
def load_phrasebank(filename):
    
    with zipfile.ZipFile(zip_path) as z:
        name = next(
            n for n in z.namelist()
            if n.lower().endswith(filename.lower())
        )
        
        raw = z.read(name).decode("iso-8859-1")

    rows = []

    for line in raw.splitlines():
        line = line.strip()

        if not line:
            continue

        sentence, label = line.rsplit("@", 1)

        rows.append({
            "sentence": sentence,
            "label": label2id[label.strip()]
        })

    return pd.DataFrame(rows)

In [6]:
df_50 = load_phrasebank("sentences_50agree.txt")
df_50.shape

(4846, 2)

In [ ]:
#How much data do we lose as we demand stronger agreement
df_66 = load_phrasebank("sentences_66agree.txt")
df_75 = load_phrasebank("sentences_75agree.txt")
df_all = load_phrasebank("sentences_allagree.txt")

print("50% agreement:", df_50.shape)
print("66% agreement:", df_66.shape)
print("75% agreement:", df_75.shape)
print("100% agreement:", df_all.shape)

50% agreement: (4846, 2)
66% agreement: (4217, 2)
75% agreement: (3453, 2)
100% agreement: (2264, 2)


In [12]:
#How much retention of the data we get with original 
# 50% agreement in each subset
retention_50 = len(df_50)/len(df_50) * 100
retention_66 = len(df_66)/len(df_50) * 100
retention_75 = len(df_75)/len(df_50) * 100
retention_all = len(df_all)/len(df_50) * 100

print("50% agreement:", retention_50)
print("66% agreement:", retention_66)
print("75% agreement:", retention_75)
print("100% agreement:", retention_all)

50% agreement: 100.0
66% agreement: 87.02022286421791
75% agreement: 71.25464300453983
100% agreement: 46.718943458522496


# Explanation 

the trade-off is:
higher agreement = more reliable labels, but less training data

Nacir specifically said not to overcomplicate this choice: pick a reasonable threshold that keeps as much usable data as possible while reducing annotation uncertainty.  

That is why this retention calculation matters.

For example, if we choose 100% agreement, you throw away more than half the available sentences:

46.7%  retained

That gives cleaner labels, but much less data for the model to learn from.

If you choose 66% agreement, you keep:

87.0% retained

of the original dataset while requiring stronger agreement than a simple 50% majority.

So at the moment, 66% looks like a very sensible compromise.

In [ ]:
#check between the agreement the split between the labels
print("50%:")
print(df_50["label"].value_counts(normalize=True).sort_index())

print("\n66%:")
print(df_66["label"].value_counts(normalize=True).sort_index())

print("\n75%:")
print(df_75["label"].value_counts(normalize=True).sort_index())

print("\n100%:")
print(df_all["label"].value_counts(normalize=True).sort_index())

50%:
label
0    0.124639
1    0.594098
2    0.281263
Name: proportion, dtype: float64

66%:
label
0    0.121888
1    0.601138
2    0.276974
Name: proportion, dtype: float64

75%:
label
0    0.121633
1    0.621489
2    0.256878
Name: proportion, dtype: float64

100%:
label
0    0.133834
1    0.614399
2    0.251767
Name: proportion, dtype: float64


### Agreement Threshold Selection

We selected the 66% annotator-agreement subset for the remainder of the project.

This subset retains approximately 87% of the observations available in the 50% agreement dataset while requiring a stronger level of agreement between annotators.

The sentiment class distribution also remains very similar to the 50% subset:

- Negative: ~12.2%
- Neutral: ~60.1%
- Positive: ~27.7%

Higher agreement thresholds substantially reduce the amount of available training data without providing a major improvement in class balance. Therefore, the 66% threshold provides a reasonable trade-off between annotation reliability and data availability.

In [ ]:
#dataset auditing
#find out if we have missing values in datasest
print(df_66.isna().sum())


sentence    0
label       0
dtype: int64


In [16]:
#find out duplicates in dataset
repeated_sentences = df_66["sentence"].duplicated().sum()
print(f"we have {repeated_sentences} repeated sentences")

we have 6 repeated sentences


In [17]:
#examine duplicates
duplicates = df_66[df_66["sentence"].duplicated(keep=False)]
duplicates.sort_values("sentence")

,sentence,label
2003,Ahlstrom 's share is quoted on the NASDAQ OMX ...,1
2004,Ahlstrom 's share is quoted on the NASDAQ OMX ...,1
2641,Proha Plc ( Euronext :7327 ) announced today (...,1
2642,Proha Plc ( Euronext :7327 ) announced today (...,1
2164,SSH Communications Security Corporation is hea...,1
2165,SSH Communications Security Corporation is hea...,1
2745,The company serves customers in various indust...,1
2746,The company serves customers in various indust...,1
959,The issuer is solely responsible for the conte...,1
960,The issuer is solely responsible for the conte...,1


### Duplicate Analysis

The selected 66% agreement subset contained no missing values.

Six duplicate occurrences were identified, corresponding to six pairs of repeated sentences (12 rows in total when including the original occurrences). All duplicated sentences had consistent sentiment labels, and all were labelled neutral(1).

Because identical sentences appearing in different train/validation/test partitions could introduce data leakage and artificially inflate model performance, one copy of each duplicated sentence was retained and the redundant copies were removed before splitting the dataset.

In [25]:
# Check whether identical sentences have conflicting labels
label_counts = (
    df_66.groupby("sentence")["label"]
         .nunique()
)

conflicting_sentences = label_counts[label_counts > 1]

print(
    "Sentences with conflicting labels:",
    len(conflicting_sentences)
)

Sentences with conflicting labels: 0


In [18]:
#remove redundant copies
df_66_clean = df_66.drop_duplicates(
    subset="sentence",
    keep="first"
).copy()

print("Before:", df_66.shape)
print("After:", df_66_clean.shape)


Before: (4217, 2)
After: (4211, 2)


### Dataset Quality Audit

The 66% agreement subset contained 4,217 observations and no missing values.

Six redundant duplicate observations were identified. In every case, the repeated sentences had consistent sentiment labels. To prevent identical observations from appearing across different train/validation/test partitions, the redundant copies were removed, leaving 4,211 unique sentences.

No identical sentences with conflicting sentiment labels were found.

In [21]:
#check for empty sentences
empty_sentences = df_66_clean["sentence"].str.strip().eq("").sum()

print("Empty sentences:", empty_sentences)

Empty sentences: 0


In [22]:
#Now we do light text cleaning
df_66_clean["sentence_clean"] = (
    df_66_clean["sentence"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
)

#We're not removing punctuation, numbers, stopwords, 
# or changing case yet. 
# Those are modeling/preprocessing choices 
# that could affect the financial meaning.

In [23]:
#final check of the dataset
df_66_clean.head()

,sentence,label,sentence_clean
0,"According to Gran , the company has no plans t...",1,"According to Gran , the company has no plans t..."
1,Technopolis plans to develop in stages an area...,1,Technopolis plans to develop in stages an area...
2,With the new production plant the company woul...,2,With the new production plant the company woul...
3,According to the company 's updated strategy f...,2,According to the company 's updated strategy f...
4,"For the last quarter of 2010 , Componenta 's n...",2,"For the last quarter of 2010 , Componenta 's n..."


In [ ]:
#check if removing the 6 duplicates did not meaningfully
# alter the class distribution
df_66_clean["label"].value_counts(normalize=True).sort_index()


label
0    0.122061
1    0.600570
2    0.277369
Name: proportion, dtype: float64

## Dataset Audit Summary

The 66% agreement subset was selected because it retains approximately 87% of
the observations from the 50% agreement dataset while providing stronger
annotator agreement.

The selected dataset initially contained 4,217 observations.

Data quality checks showed:
- 0 missing values
- 0 empty sentences
- 6 redundant duplicate observations
- 0 identical sentences with conflicting sentiment labels

The duplicate observations were removed to reduce the risk of identical
sentences appearing across train, validation, and test partitions.

After deduplication, the dataset contains 4,211 unique sentences.

Only minimal text cleaning was applied by removing leading/trailing whitespace
and collapsing repeated whitespace. Punctuation, numbers, capitalization,
stopwords, and other linguistic information were preserved because these may
contain important financial sentiment information.

The class distribution remained effectively unchanged after deduplication.